# Meal Planning Agent — Prompt Experiments

Scratchpad for iterating on prompts before building the app.

## Initialization

In [ ]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings, OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
import os
import asyncio
from IPython.display import display, Markdown

load_dotenv(override=True)

def assertKeyExists(key :str) -> str:
    value = os.getenv(key)
    if value:
        return value
    raise ValueError(f"{key} not found")

openai_api_key = assertKeyExists("OPENAI_API_KEY")
google_api_key = assertKeyExists("GOOGLE_API_KEY")
grok_api_key = assertKeyExists("GROK_API_KEY")

print("API keys loaded ✔")

high_effort_model = "gpt-5.6-sol"
balanced_model = "gpt-5.6-terra"
low_effort_model = "gpt-5.6-luna"
default_model = low_effort_model

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.6-flash", openai_client=gemini_client)

GROK_BASE_URL = "https://api.x.ai/v1"
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
grok_model = OpenAIChatCompletionsModel(model="grok-4.5", openai_client=grok_client)

print("Models loaded ✔")

base_system_instructions = '''
You are a meal planning assistant who helps people plan their meals for the week.
You are an expert on simple meals that require minimal amounts of preparation, reheat well, and are delicious.
'''

def to_markdown_list(data: list[any], bullet: str ="-"):
    """
    Converts a list into a markdown bulleted list string.
    """
    return "\n".join(f"{bullet} {str(item)}" for item in data)

def filter_out_invalid_strings(input_list: list[str], invalid_strings: set[str]) -> list[str]:
    return [item for item in input_list if item not in invalid_strings]

def clamp(n, min_n, max_n):
    return max(min_n, min(n, max_n))

def print_break():
    print("\n\n\n========================================================================================\n\n\n")

print("Utils loaded ✔")

In [ ]:
pushover_user = assertKeyExists("PUSHOVER_USER")
pushover_token = assertKeyExists("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def send_push_notification(message: str):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

## User Preferences 
Collects dietary restrictions, likes/dislikes, meals that they're tired of, and what kind of cooking equipment they have. 

In [279]:
class UserPreferences(BaseModel):
    number_of_meals: int = Field(default = 2, description="The number of meals that you should plan. Valid range of values is 1 to 10.")
    number_of_servings_per_meal: int = Field(default = 4, description="Determines how many serving portions we should make for each meal.  Valid range of values is 1 to 100.")
    dietary_restrictions: list[str] = Field(default_factory=list, description="A list of any dietary restrictions the user has that need to be considered for the meal plan.", examples=["Gluten-free", "vegetarian", "dairy-free"])
    likes: list[str] = Field(default_factory=list, description="A list of foods user's favorite foods.")
    dislikes: list[str] = Field(default_factory=list, description="A list of foods that the user does not like.")
    nutritional_goals: str = Field(default="", description="A description of the nutritional goals that the user aims to achieve with this meal plan.", examples=["Increase protein intake, lose weight, lower cholesterol"])
    meals_to_avoid_this_time: list[str] = Field(default_factory=list, description="Specific foods that the user would prefer to avoid this plan.")
    preferred_cooking_methods: list[str] = Field(default = ["oven", "stovetop", "microwave"], description="When cooking is required, these are the user's preferred methods.", examples = ["oven", "stovetop", "microwave", "grill"])
    notes: str = Field(default="", description="A paragraph of notes about any preferences that don't apply to one of the other fields.", examples=["Half of my meals should be meatless."])

class UserPreferencesReview(BaseModel):
    preferences: UserPreferences = Field(description="The user's updated preferences.")
    user_has_confirmed_that_preferences_are_correct: bool = Field(description="True only when the user has reviewed their preferences, confirmed that they are correct, and that they don't want to modify it any further.")
    follow_up_response: str = Field(description="Your polite response to the user's message confirming that you understand their request. If the user has not yet confirmed that the preferences are complete, ask them if there is anything else.")

saved_user_preferences = UserPreferences()

def set_user_preferences(update: UserPreferences):
    global saved_user_preferences
    saved_user_preferences = sanitize_user_preferences(update)

def get_user_preferences() -> UserPreferences:
    return saved_user_preferences

user_preferences_system_instructions = f'''
{base_system_instructions}
'''

user_preferences_prompter = Agent(
    name="User Preferences",
    instructions=user_preferences_system_instructions,
    model=default_model,
)

user_preferences_reviewer = Agent(
    name="User Preferences",
    instructions=user_preferences_system_instructions,
    model=default_model,
    output_type=UserPreferencesReview,
)

async def review_known_user_preferences() -> str:
    prompt =f'''
    Here is all of the information we have about the user's meal plan preferences:
    {get_user_preferences()}

    Present it to the user in an easily reviewable format.

    You need the correct preferences to make sure that the meal plan works for the user. It's essential for you to do your job correctly.
    Explain the importance to the user.
    Then ask the user to either confirm that everything looks correct or make changes.
    '''

    return (await Runner.run(user_preferences_prompter, prompt)).final_output

async def update_user_preferences(user_update: str) -> UserPreferencesReview:
    prompt =f'''
    Here is all of the information we have about the user's meal plan preferences:
    {get_user_preferences()}

    You presented this information to the user.

    The user sent this message:
    {user_update}

    If the user has not yet confirmed that the preferences are correct and complete, prompt them to finish it in the follow_up_response field.
    '''

    update = (await Runner.run(user_preferences_reviewer, prompt)).final_output
    set_user_preferences(update.preferences)
    return update

def sanitize_user_preferences(raw: UserPreferences) -> UserPreferences:
    return UserPreferences(
        number_of_meals = clamp(raw.number_of_meals, 1, 10),
        number_of_servings_per_meal = clamp(raw.number_of_servings_per_meal, 1, 100),
        dietary_restrictions = raw.dietary_restrictions,
        likes = raw.likes,
        dislikes = raw.dislikes,
        nutritional_goals = raw.nutritional_goals,
        meals_to_avoid_this_time = raw.meals_to_avoid_this_time,
        notes = raw.notes,
    )


In [ ]:
# with trace("Empty User Preferences Demo"):
#     set_user_preferences(UserPreferences())
#     print(await review_known_user_preferences())

In [ ]:
# with trace("Happy Path User Preferences Demo"):
#     set_user_preferences(UserPreferences(dietary_restrictions=["egg allergy"],nutritional_goals="Lose weight, High protein",notes="I want half my meals to be meatless"))
#     print(await review_known_user_preferences())
#     print("\n\n")

#     first_user_response = "I want half of my meals to be meatless. I want my food to be relatively healthy too so I can try to lose wait as well."
#     print(f"{first_user_response}\n\n")
#     first_update = await update_user_preferences(first_user_response)
#     print(f"{first_update}\n\n")
    

#     second_user_response = "That looks correct, thank you."
#     print(f"{second_user_response}\n\n")
#     second_update = await update_user_preferences(second_user_response)
#     print(f"{second_update}\n\n")


## Meal Generation

### Meal Brainstorming Agent

An agent that generates a bunch of meal ideas.

In [ ]:
from datetime import datetime

# Used to make the meals weather-appropriate and add a little bit of differentiation to the prompt week-to-week.
def get_seasonal_report():
    now = datetime.now()
    month_name = now.strftime("%B")
    month_num = now.month
    
    # Season list (indexed 0-3)
    seasons = ["Winter", "Spring", "Summer", "Autumn"]
    
    # Weather descriptions for each season
    weather_data = {
        "Winter": "Expect cold temperatures, frosty mornings, and the occasional flurry of snow.",
        "Spring": "The days are getting longer and you'll see flowers beginning to bloom.",
        "Summer": "It's time for sunshine, warm breeze, and plenty of outdoor activities.",
        "Autumn": "The air is turning crisp and the leaves are putting on a colorful show."
    }
    
    # The math trick: (month % 12 // 3)
    # Dec(12), Jan(1), Feb(2) map to 0 (Winter)
    # Mar(3), Apr(4), May(5) map to 1 (Spring)
    # Jun(6), Jul(7), Aug(8) map to 2 (Summer)
    # Sep(9), Oct(10), Nov(11) map to 3 (Autumn)
    season_idx = (month_num % 12 // 3)
    season = seasons[season_idx]
    description = weather_data[season]
    
    return f"It's {month_name} and {season} is here! {description}"

# Output the result
print(get_seasonal_report())

In [ ]:
import random

# Randomly returns a model so that the behavior is more variable
def get_random_model():
    models = [default_model, gemini_model, grok_model]
    return random.choice(models)

In [ ]:
class PreparedDish(BaseModel):
    name: str = Field(description="The short name of a dish.")
    description: str = Field(description="A 1-2 sentence description of the dish.")
    special_diet_labels: list[str] = Field(description="A list of any dietary restrictions that this meal satisfies", examples=["vegetarian", "gluten-free"])
    category: list[str] = Field(description="The category of food.", examples = ["Italian", "Chinese"])


class MealPlanIdeas:
    entree_ideas: list[PreparedDish]
    side_ideas: list[PreparedDish] 

    def __init__(self, entree_ideas: list[PreparedDish], side_ideas: list[PreparedDish]) -> None:
        self.entree_ideas = entree_ideas
        self.side_ideas = side_ideas

    def __str__(self):
        return f"entrees: {self.entree_ideas}, sides: {self.side_ideas}"


brainstorm_instructions = f'''
{base_system_instructions}

Prioritize meals that can be made with minimal (less than ten) unique ingredients.

{get_seasonal_report()}
Try to pick meals that are popular for this time of year.
'''

meal_brainstorming_agent = Agent(
    name="Meal Brainstormer",
    instructions=brainstorm_instructions,
    model=get_random_model(),
    output_type=list[PreparedDish],
)

async def generate_meal_ideas(prompt: str) -> list[PreparedDish]:
    return (await Runner.run(meal_brainstorming_agent, prompt)).final_output


async def create_meal_plan_brainstorm(number_of_meals: int) -> MealPlanIdeas:
    number_of_meals = clamp(number_of_meals, 1, 10)
    preferences = get_user_preferences()
    user_preferences_prompt = f'''
    Here is the user's preferences:
    {preferences}
    '''
    
    meal_idea_multiple = 5 # generate extra ideas to allow for more randomness and also in case we need to drop some of them during validation
    entrees, sides = await asyncio.gather(
        generate_meal_ideas(f"Suggest {number_of_meals * meal_idea_multiple} different entrees. {user_preferences_prompt}"),
        generate_meal_ideas(f"Suggest {number_of_meals * meal_idea_multiple} different sides. {user_preferences_prompt}")
    )
    return MealPlanIdeas(
        entree_ideas = entrees, 
        side_ideas = sides,
    )

In [ ]:
# with trace("Meal Brainstorming Test"):
#     user_prefs = UserPreferences(nutritional_goals="Lose weight, High protein",notes="I want half my meals to be meatless")
#     set_user_preferences(user_prefs)
#     unfiltered_meal_plan = await create_meal_plan_brainstorm(user_prefs.number_of_meals)
#     print(unfiltered_meal_plan)

### Validation
Filters brainstorm ideas that don't conform to likes, dislikes, and user preferences.
Verifies that it's not too repetitive with the meals from last week.

In [ ]:
meal_validation_instructions = f'''
{base_system_instructions}

Part of your job is inspecting menus for clients and flagging any foods that they would dislike.
Identifying violations of your client's food allergen or dietary restriction rules is your highest priority.
'''

def filter_out_flagged_dishes(dishes: list[PreparedDish], flagged_names: set[str]) -> list[PreparedDish]:
    return [dish for dish in dishes if dish.name not in flagged_names]

meal_filterer = Agent(
    name = "Meal Idea Filterer",
    instructions=meal_validation_instructions,
    model = default_model,
    output_type = list[str],
)

async def filter_meal_ideas(meal_ideas: MealPlanIdeas) -> MealPlanIdeas:
    all_dishes = meal_ideas.entree_ideas + meal_ideas.side_ideas
    prompt = f'''
    You have a list of foods that have been generated as candidates for the user's meal plan:
    {to_markdown_list([dish.name for dish in all_dishes])}

    Here is the user's meal plan preferences:
    {get_user_preferences()}

    Your job is to identify and return the exact dish names from the list above that are poor candidates based on the user's preferences.
    '''
    flagged_foods = set((await Runner.run(meal_filterer, prompt)).final_output)

    # Filter out flagged foods and shuffle them to make the selection more random.
    return MealPlanIdeas(
        entree_ideas=filter_out_flagged_dishes(meal_ideas.entree_ideas, flagged_foods),
        side_ideas=filter_out_flagged_dishes(meal_ideas.side_ideas, flagged_foods),
    )

In [ ]:
# with trace("Meal Validation Test"):
#     set_user_preferences(UserPreferences(dietary_restrictions=["gluten-free"]))
#     result = await filter_meal_ideas(
#         MealPlanIdeas(
#             entree_ideas=[
#                 PreparedDish(name = "Spaghetti and Meatballs", description = "", special_diet_labels = [], category = []),
#                 PreparedDish(name = "Chicken Thighs", description = "", special_diet_labels = ["gluten-free"], category = []),
#             ],
#             side_ideas=[
#                 PreparedDish(name = "Macaroni and Cheese", description = "", special_diet_labels = [], category = []),
#                 PreparedDish(name = "Salad", description = "", special_diet_labels = ["gluten-free"], category = []),
#             ],
#         )
#     )
#     print(result)

### Pairing
Picks meals and attempts to pair it with similar side based on category.

In [ ]:
import random

class MealPairing(BaseModel):
    entree: PreparedDish = Field(description="main entree")
    side: PreparedDish = Field(description="side")

pairing_system_instructions=f'''
{base_system_instructions}
'''
entree_picking_agent=Agent(
    name="Entree Picking Agent",
    instructions=pairing_system_instructions,
    model=default_model,
    output_type=list[PreparedDish]
)
pairing_agent=Agent(
    name="Meal Pairing Agent",
    instructions=pairing_system_instructions,
    model=default_model,
    output_type=list[MealPairing]
)

meal_choice_validation_agent=Agent(
    name="Meal Choice Validation Agent",
    instructions=pairing_system_instructions,
    model=gemini_model,
    output_type=bool
)

async def generate_meals(brainstorm_results: MealPlanIdeas, number_of_meals: int) -> list[MealPairing]:
    attempts = 0
    while(True):
        attempts += 1
        entree_choices = await pick_entrees(number_of_meals = number_of_meals, entrees = brainstorm_results.entree_ideas)
        meal_choices = await pair_with_sides(entrees = entree_choices, sides = brainstorm_results.side_ideas)
        if (await validate_meal_choices(meal_choices) or attempts > 3):
            return meal_choices

async def pick_entrees(number_of_meals: int, entrees: list[PreparedDish]) -> list[PreparedDish]:
    # Shuffle to make meal selection more unpredictable
    shuffled_entrees = random.sample(entrees, len(entrees))
    prompt=f'''
    Your job is to pick {number_of_meals} entree(s) for the user's meal plan.

    You can choose from the following list:
    {to_markdown_list(shuffled_entrees)}

    Make sure that your final selection conforms to the user's preferences:
    {get_user_preferences()}

    Make sure that your {number_of_meals} selection(s) are different categories from each other.
    '''
    return (await Runner.run(entree_picking_agent, prompt)).final_output


async def pair_with_sides(entrees: list[PreparedDish], sides: list[PreparedDish]) -> list[MealPairing]:
    # Shuffle to make meal selection more unpredictable
    shuffled_sides = random.sample(sides, len(sides))
    prompt = f'''
    You're writing a meal plan for the user.

    You already have the entrees picked out:
    {to_markdown_list(entrees)}

    Now you need to pair those entrees with sides so that you have a complete meal.
    You can choose from the following list of sides:
    {to_markdown_list(shuffled_sides)}

    For each entree, try to pick a side that compliments it.
    This means that they should be in the same or similar categories.
    The entree and side shouldn't be the same dish though.
    '''
    return (await Runner.run(pairing_agent, prompt)).final_output

async def validate_meal_choices(meals: list[MealPairing]) -> bool:
    prompt = f'''
    You're writing a meal plan for the user.

    You've picked meal(s) for the meal plan':
    {to_markdown_list(meals)}

    Determine if that list conforms to the user's preferences:
    {get_user_preferences()}
    '''
    return (await Runner.run(meal_choice_validation_agent, prompt)).final_output

async def generate_meal_pairings_for_meal_plan() -> list[MealPairing]:
    number_of_meals = get_user_preferences().number_of_meals
    return await generate_meal_pairings(number_of_meals=number_of_meals)

async def generate_meal_pairings(number_of_meals: int) -> list[MealPairing]:
    number_of_meals = clamp(number_of_meals, 1, 10)
    brainstorm_results = await create_meal_plan_brainstorm(number_of_meals = number_of_meals)
    return await generate_meals(brainstorm_results = brainstorm_results, number_of_meals = number_of_meals)


### User Feedback
Asks user to approve the selected meals. Can make changes to a meal or select a different one from the brainstorming list. If all else fails, can escape back and restart with a different model.

In [ ]:
meal_plan_feedback_system_prompt =f'''
{base_system_instructions}
'''
meal_plan_feedback_agent = Agent(
    name = "Meal Plan Feedback Agent",
    instructions = meal_plan_feedback_system_prompt,
    model = default_model,
)

async def present_meals_for_feedback(meal_pairings: list[MealPairing]) -> str:
    prompt=f'''
    You have successfully generated meals for the user's meal plan!
    Here they are:
    {to_markdown_list(meal_pairings)}

    Present all of them to the user in markdown for their review.

    For your tone, be polite and don't be afraid to embellish how tasty these meals are going to be.
    Ask them to approve the meals or request changes before you begin generating recipes.
    '''

    return (await Runner.run(meal_plan_feedback_agent, prompt)).final_output

### Demo
Ties the meal generation flow altogether

In [ ]:
# with trace("Meal Plan Generation Demo"):
#     set_user_preferences(
#         UserPreferences(
#             nutritional_goals="Lose weight, Increase protein intake",
#             notes="I want half my meals to be meatless",
#         )
#     )
#     meal_pairings = await generate_meal_pairings_for_meal_plan()
#     print(await present_meals_for_feedback(meal_pairings))

## Recipes

### Generation
Writes recipes for the selected meals

In [ ]:
from dataclasses import dataclass

grocery_departments = ["Produce, Bakery, Pantry, Meat, Refrigerated, Dairy, Frozen, Pharmacy, Other"]

class Ingredient(BaseModel):
    name: str = Field(description = "The name of the ingredient.", examples=["Minced Garlic"])
    unit: str = Field(description = "The unit type by which the ingredient is measured in the recipe.", examples=["Tablespoon"])
    quantity: float = Field(description = "The quanitity of units used in the recipe.", examples=[1.0])
    grocery_store_department: str = Field(
        description = f"The area of the grocery store where this ingredient can be found. Here are the valid values: {grocery_departments}"
    )

class Recipe(BaseModel):
    ingredients: list[Ingredient] = Field(description="A list of incredients for the recipe so the user can add them to shopping list.")
    number_of_servings: int = Field(description = "How many servings the recipe makes.")
    cooking_instructions: str = Field(description="A list of instructions for how to prepare and cook the entree, written in markdown.")

@dataclass
class MealPlanItem:
    entree: PreparedDish
    entree_recipe: Recipe
    side: PreparedDish
    side_recipe: Recipe


recipe_generation_system_instructions = f'''
{base_system_instructions}
'''
recipe_generation_agent = Agent(
    name = "Recipe Generation Agent",
    instructions = recipe_generation_system_instructions,
    model = default_model,
    output_type = Recipe,
)
recipe_adjustment_agent = Agent(
    name = "Recipe Generation Agent",
    instructions = recipe_generation_system_instructions,
    model = high_effort_model,
    output_type = Recipe,
)
recipe_validation_agent = Agent(
    name = "Recipe Generation Agent",
    instructions = recipe_generation_system_instructions,
    model = grok_model,
    output_type=bool,
)

async def generate_recipes(meals: list[MealPairing]) -> list[MealPlanItem]:
    tasks = [generate_recipes_for_meal(item) for item in meals]
    meal_plan_items = await asyncio.gather(*tasks)
    return meal_plan_items

async def generate_recipes_for_meal(meal: MealPairing) -> MealPlanItem:
    entree_recipe, side_recipe = await asyncio.gather(
        generate_recipe(meal.entree),
        generate_recipe(meal.side),
    )
    return MealPlanItem(
        entree = meal.entree,
        entree_recipe=entree_recipe,
        side = meal.side,
        side_recipe = side_recipe,
    )

async def generate_recipe(dish: PreparedDish) -> Recipe:
    prompt = f'''
    The user has selected the following meal for their meal plan:
    {dish}

    I want you to generate a recipe for the above. Break it into 3 sections: Ingredients, Preparation Instructions, Cooking Instructions

    Make sure that the recipe conforms to the user's preferences:
    {get_user_preferences()}

    The recipe should use less than 10 ingredients and preparation time under 20 minutes.
    '''
    attempts = 0
    while(True):
        attempts += 1
        recipe = (await Runner.run(recipe_generation_agent, prompt)).final_output
        recipe = await adjust_for_servings_count_if_necessary(recipe)
        passes_validation = await validate_recipe(dish, recipe)
        if (passes_validation or attempts > 3):
            return recipe

async def adjust_for_servings_count_if_necessary(recipe: Recipe) -> Recipe:
    target_servings = get_user_preferences().number_of_servings_per_meal
    if (recipe.number_of_servings == target_servings):
        return recipe
    prompt = f'''
    You've generated a recipe for a meal that makes {recipe.number_of_servings}.
    However, the user has explicitly mentioned that they want to make {target_servings}.
    That means each of the ingredient quantities need to be multiplied by a factor of {target_servings / recipe.number_of_servings}

    Please adjust the recipe (seen below) so that it makes the correct number of servings:
    {recipe}
    '''
    return (await Runner.run(recipe_adjustment_agent, prompt)).final_output

async def validate_recipe(dish: PreparedDish, recipe: Recipe) -> bool:
    prompt = f'''
    You're writing a meal plan for the user and they've selected the following dish:
    {dish.name}

    You've written the following recipe for that dish:
    {recipe}

    Return true if the recipe is simple and conforms to the user's preferences:
    {get_user_preferences()}
    '''
    return (await Runner.run(recipe_validation_agent, prompt)).final_output

In [ ]:
# with trace("Recipe Generation Demo"):
#     set_user_preferences(
#         UserPreferences(
#             nutritional_goals="Lose weight, Increase protein intake",
#             notes="I want half my meals to be meatless",
#         )
#     )
#     meal_pairings = await generate_meal_pairings_for_meal_plan()
#     meal_plan_items = await generate_recipes(meal_pairings)
#     for meal_plan_item in meal_plan_items:
#         display(Markdown(to_markdown_list(meal_plan_item.entree_recipe.ingredients)))
#         print("\n\n")
#         display(f"servings size: {meal_plan_item.entree_recipe.number_of_servings}")
#         print("\n\n")
#         display(Markdown(meal_plan_item.entree_recipe.cooking_instructions))
#         print("\n\n---------------------------------\n\n")
#         display(Markdown(to_markdown_list(meal_plan_item.side_recipe.ingredients)))
#         print("\n\n")
#         display(f"servings size: {meal_plan_item.side_recipe.number_of_servings}")
#         print("\n\n")
#         display(Markdown(meal_plan_item.side_recipe.cooking_instructions))
#         print("\n\n")

## Shopping List

### Aggregation
Collects all of the ingredients from the recipes above

In [ ]:
def get_consolidated_ingredients(meal_plan: list[MealPlanItem]) -> list[Ingredient]:
    """
    Extracts all ingredients and sums the quantities of items 
    with the same name and unit.
    """
    totals = {} # Key: (name, unit, department), Value: total_quantity

    for item in meal_plan:
        # Helper to process recipe lists
        recipe_ingredients = item.entree_recipe.ingredients + item.side_recipe.ingredients
        
        for ing in recipe_ingredients:
            # Create a unique key based on name and unit
            # We use lower() to ensure "Garlic" and "garlic" match
            key = (ing.name.lower(), ing.unit.lower(), ing.grocery_store_department)
            
            if key in totals:
                totals[key] += ing.quantity
            else:
                totals[key] = ing.quantity

    # Convert the dictionary back into a list of Ingredient objects
    consolidated = [
        Ingredient(
            name=name.title(), 
            unit=unit.title(), 
            quantity=qty, 
            grocery_store_department=dept
        )
        for (name, unit, dept), qty in totals.items()
    ]
    
    return consolidated

def sort_ingredients(ingredients: list[Ingredient]) -> list[Ingredient]:    
    order_map = {dept: i for i, dept in enumerate(grocery_departments)}

    def sorting_key(item: Ingredient):
        # Primary key: the index from our map. 
        # .get() handles cases where a department might not be in the list (defaults to the end)
        department_rank = order_map.get(item.grocery_store_department, len(grocery_departments))
        
        # Secondary key: the name (alphabetical)
        return (department_rank, item.name.lower())

    return sorted(ingredients, key=sorting_key)

def generate_ingredients_markdown(ingredients: list[Ingredient]) -> str:
    """
    Takes a sorted list of Ingredient objects and returns a 
    Markdown-formatted string grouped by department.
    """
    lines = ["# Grocery List"]
    current_dept = None

    for item in ingredients:
        # Check if we have moved to a new department
        if item.grocery_store_department != current_dept:
            current_dept = item.grocery_store_department
            # Add an extra newline for spacing between sections
            lines.append(f"\n## {current_dept}")
        
        # Add the checklist item
        # :g format removes trailing zeros (e.g., 1.0 -> 1)
        lines.append(f"- [ ] {item.quantity:g} {item.unit} {item.name}")

    return "\n".join(lines)


## Meal Plan Presentation Agent
Turns everything into pretty markdown:

In [ ]:
@dataclass
class MealPlan:
    plan: str
    shopping_list: str

async def generate_meal_plan(meal_pairings: list[MealPairing]) -> MealPlan:
    meal_plan_items = await generate_recipes(meal_pairings)
    return MealPlan(
        plan = await write_meal_plan(meals = meal_plan_items), 
        shopping_list=write_shopping_list(meals = meal_plan_items),
    )

author_instructions = f'''
{base_system_instructions}
'''
author_agent = Agent(
    name="Meal Plan Author",
    model = default_model,
    instructions=author_instructions
)

async def write_meal_plan(meals: list[MealPlanItem]) -> str:
    prompt = f'''
    You have generated the following meal plan items:
    {meals}

    I want you to write a pretty Markdown string. It should start off with a high level summary of the dishes that are included in the meal plan.

    Then there should be a divider followed by a detailed section specific to each PreparedDish.
    Each should include an ingredients segment and a cooking instructions segment.
    '''
    return (await Runner.run(author_agent, prompt)).final_output

def write_shopping_list(meals: list[MealPlanItem]) -> str:
    ingredients = get_consolidated_ingredients(meal_plan=meals)
    sorted_ingredients = sort_ingredients(ingredients)
    return generate_ingredients_markdown(sorted_ingredients)

## Orchestration
Ties everything together and runs it in a UI.

### Basic Sequential Iteration
Goes through steps one by one.

In [ ]:
with trace("Meal Planning - Full Demo"):
    set_user_preferences(UserPreferences())
    print(await review_known_user_preferences())
    print_break()

    first_user_response = "I want half of my meals to be meatless. My goals are to lose weight and increase protein intake."
    first_update = await update_user_preferences(first_user_response)
    print(first_update.follow_up_response)
    print_break()

    second_user_response = "That looks correct, thank you."
    second_update = await update_user_preferences(second_user_response)
    print(second_update.follow_up_response)
    print_break()

    meal_pairings = await generate_meal_pairings_for_meal_plan()
    print(await present_meals_for_feedback(meal_pairings))
    print_break()

    meal_plan = await generate_meal_plan(meal_pairings=meal_pairings)
    display(Markdown(meal_plan.shopping_list))
    print_break()
    display(Markdown(meal_plan.plan))

### LLM Orchestration
Use tools and subagents to allow better flexibility like moving backwards to change preferences or meal choices